# Day 2 — Computer Vision & Perception for UR5

Welcome to Day 2 of the **UR5 AI Workshop**!

### 🎯 Learning Objectives
By the end of this notebook, you will:
1. Capture and process images from a camera.
2. Apply computer vision algorithms (edges, contours, color filtering).
3. Use pre-trained AI models for object detection.
4. Map image coordinates to robot coordinates.


## 🧩 Exercise 1 — Capture and Display Images

In [ ]:
# Student Task:
import cv2

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow('UR5 Camera Feed', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.imwrite('snapshot.jpg', frame)
        break
cap.release()
cv2.destroyAllWindows()

## 🧩 Exercise 2 — Basic Image Processing (Edges & Contours)

In [ ]:
# Student Task:
import cv2
import numpy as np

img = cv2.imread('snapshot.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 100, 200)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(img, contours, -1, (0,255,0), 2)
cv2.imshow('Contours', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 🧩 Exercise 3 — Object Detection using a Pre-trained Model

In [ ]:
# Student Task:
import cv2
import numpy as np

net = cv2.dnn.readNetFromCaffe('MobileNetSSD_deploy.prototxt', 'MobileNetSSD_deploy.caffemodel')
classes = ['background','aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair',
           'cow','diningtable','dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor']

img = cv2.imread('snapshot.jpg')
h, w = img.shape[:2]
blob = cv2.dnn.blobFromImage(cv2.resize(img, (300,300)), 0.007843, (300,300), 127.5)
net.setInput(blob)
detections = net.forward()
for i in range(detections.shape[2]):
    confidence = detections[0,0,i,2]
    if confidence > 0.4:
        idx = int(detections[0,0,i,1])
        box = detections[0,0,i,3:7]*np.array([w,h,w,h])
        (startX, startY, endX, endY) = box.astype('int')
        label = f"{classes[idx]}: {confidence*100:.1f}%"
        cv2.rectangle(img, (startX, startY), (endX, endY), (0,255,0), 2)
        cv2.putText(img, label, (startX, startY-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
cv2.imshow('Detections', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 🧩 Exercise 4 — Pixel to Robot Coordinate Mapping

In [ ]:
# Student Task:
def pixel_to_robot(px, py, scale=0.5, offset=(0.3, -0.2)):
    X = offset[0] + px * scale / 1000
    Y = offset[1] + py * scale / 1000
    return X, Y

print(pixel_to_robot(200, 150))

## 🎓 Final Challenge — Vision-Guided Pick Simulation
Combine all steps:
1. Detect an object (color or model-based).
2. Compute its pixel center.
3. Convert to robot coordinates.
4. Send a move command to URSim to align the TCP above the object.

**Hint:** Reuse your `pixel_to_robot()` and UR RTDE control interface.